In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

print(bool(os.getenv("GROQ_API_KEY")))

True


In [26]:
from langchain_groq import ChatGroq

llm_discovery = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [11]:
import pandas as pd
amazon_pairs=pd.read_csv("amazon_pairs.csv")

In [16]:
print(amazon_pairs.columns.tolist())

['customer_message', 'amazon_response']


In [18]:
sample_df = amazon_pairs[
    ["customer_message"]
].dropna().sample(
    n=min(150, len(amazon_pairs)),
    random_state=42
)

messages = "\n".join(
    f"{i+1}. {msg}"
    for i, msg in enumerate(sample_df["customer_message"])
)

prompt = f"""
You are analyzing real customer-support conversations from Amazon.

Below are {len(sample_df)} representative customer messages.

Discover the SMALL number of recurring customer-support intent
categories present in this dataset.

Rules:
- Base categories ONLY on the provided messages.
- Group semantically similar requests together.
- Prefer 8-15 broad but distinct categories.
- Each category should represent a different customer goal/problem.
- Do not create a category for every small variation.
- Give each category a short name.
- Give a one-sentence description.
- Give 2 example messages from the supplied data.
- Do NOT label every message yet.

Return ONLY valid JSON:

[
  {{
    "intent": "Short Intent Name",
    "description": "What this intent represents",
    "examples": [
      "example customer message",
      "example customer message"
    ]
  }}
]

CUSTOMER MESSAGES:

{messages}
"""

response = llm_discovery.invoke(prompt)

print(response.content)

[
  {
    "intent": "Delivery Problems",
    "description": "Customer reports that a package was not delivered on time, delivered to the wrong place, or otherwise mishandled.",
    "examples": [
      "Ayer no me entregaron un paquete de @116928 por ausente a las 18:35. … ES UNA PUTA VERGUENZA！",
      "Dear Amazon, Where is my leaf blower!!!"
    ]
  },
  {
    "intent": "Order Tracking & Status",
    "description": "Customer asks for an update on the whereabouts of an order or cannot find tracking information.",
    "examples": [
      "Order No 407-6706063-7981118. God know when will you guys delivery. Terrible Service, no one knows the status.",
      "the tracking begins with TBA - i don't even know what courier that is lol"
    ]
  },
  {
    "intent": "Returns & Refunds",
    "description": "Customer wants to return an item, is waiting for a refund, or is confused about the return process.",
    "examples": [
      "item not delivered,customer service said to return item for a r

In [19]:
# Get Groq's discovered taxonomy
discovered_intents = response.content

print(discovered_intents)

[
  {
    "intent": "Delivery Problems",
    "description": "Customer reports that a package was not delivered on time, delivered to the wrong place, or otherwise mishandled.",
    "examples": [
      "Ayer no me entregaron un paquete de @116928 por ausente a las 18:35. … ES UNA PUTA VERGUENZA！",
      "Dear Amazon, Where is my leaf blower!!!"
    ]
  },
  {
    "intent": "Order Tracking & Status",
    "description": "Customer asks for an update on the whereabouts of an order or cannot find tracking information.",
    "examples": [
      "Order No 407-6706063-7981118. God know when will you guys delivery. Terrible Service, no one knows the status.",
      "the tracking begins with TBA - i don't even know what courier that is lol"
    ]
  },
  {
    "intent": "Returns & Refunds",
    "description": "Customer wants to return an item, is waiting for a refund, or is confused about the return process.",
    "examples": [
      "item not delivered,customer service said to return item for a r

In [21]:
#fine tuning the intents
cleanup_prompt = f"""
You are designing the intent taxonomy for an Amazon customer-support AI agent.

The following intent categories were discovered from real Amazon customer
messages:

{discovered_intents}

Create the FINAL taxonomy.

Rules:
1. Keep the taxonomy small: preferably 8-12 intents.
2. Merge overlapping categories.
   Example: "Delivery Problems" + "Order Tracking & Status"
   should normally become "Delivery / Tracking".
3. Remove categories that are too specific or do not represent a
   meaningful customer-support intent.
4. Do not invent completely new categories unless necessary.
5. Every intent must represent a distinct customer goal or problem.
6. Names must be short and consistent.
7. Include a clear description for every intent.
8. Do not include examples in the final output.
9. The taxonomy will be used to label thousands of future messages,
   so categories must be broad enough to generalize.

Return ONLY valid JSON in this format:

[
  {{
    "intent": "Intent Name",
    "description": "Clear description of when this intent should be used."
  }}
]
"""

final_response = llm_discovery.invoke(cleanup_prompt)

print(final_response.content)

[
  {
    "intent": "Delivery & Tracking",
    "description": "Customer reports a missing, late, or mis‑delivered package, or asks for the current status or tracking details of an order."
  },
  {
    "intent": "Returns & Refunds",
    "description": "Customer wants to return an item, is waiting for a refund, or has questions/issues with the return or refund process."
  },
  {
    "intent": "Account Access & Security",
    "description": "Issues related to logging in, password resets, account lockouts, or suspicious activity on the customer's Amazon account."
  },
  {
    "intent": "Benefits & Promotions",
    "description": "Questions about Prime membership features, eligibility, changes to benefits, or inquiries/complaints about promotional offers, contests, and giveaways."
  },
  {
    "intent": "Digital Content & Device Issues",
    "description": "Problems downloading, streaming, or using Amazon digital services (Kindle, Audible, Prime Video, etc.) or Amazon‑branded devices (Echo,

In [23]:
#these are the intents that we have discovered
INTENTS = {
    "Delivery & Tracking":
        "Missing, late, mis-delivered packages, order status, or tracking questions.",

    "Returns & Refunds":
        "Returns, refunds, refund delays, or problems with the return/refund process.",

    "Account Access & Security":
        "Login, password, account access, account lockout, or suspicious account activity.",

    "Prime Membership & Benefits":
        "Questions or problems related to Prime membership, subscription, eligibility, or Prime benefits.",

    "Promotions & Offers":
        "Questions or problems involving discounts, promotional offers, contests, giveaways, or promotional eligibility.",

    "Digital Content & Device Issues":
        "Problems with Kindle, Audible, Prime Video, digital content, Echo, Fire TV, or other Amazon devices/services.",

    "Product Quality & Listing":
        "Damaged, defective, incomplete, incorrect, or mis-described products and inaccurate listings.",

    "Payment & Billing":
        "Incorrect charges, payment problems, cashback, Amazon Pay, billing disputes, or payment-related issues.",

    "Customer Service Experience":
        "Complaints or feedback about Amazon support, response times, unhelpful agents, or difficulty reaching support.",

    "Employment & Flex":
        "Questions about Amazon Flex, jobs, hiring, or employment programs."
}

In [25]:
amazon_pairs.shape

(168823, 2)